Lecture 4 2025 Probabilistic ML Hennig 2025

Based upon code shown by Hennig in [[1,57:27]](#References)

In [ ]:
from gaussians.gaussians_lecture_04 import *
from jax import numpy as jnp
from jax import random as jrnd
from jaxtyping import Array, Float
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap  # Had to add this myself to get LSC

ob = LinearSegmentedColormap.from_list('ob', [rgb.tue_blue, rgb.tue_orange], N=1024)

ModuleNotFoundError: No module named 'gaussians_lecture_04'

In [ ]:
# From [1,58:08]
def GEQRF(A: Float[Array, "M N"], prior: Gaussian) -> (Callable, Float[Array, "M M"], Float[Array, "M N"], Float[Array, "M M"]):
    M , N = A.shape # dimnesions
    U, R, nu = (
        jnp.zeros((M, N)),
        jnp.zeros((N, N)),
        jnp.zeros(N),
    ) # Storage
    Sigma = prior.Sigma
    for n in range(N): # matrix decomposition
        an = A[:, n]
        un = Sigma @ an
        un = un / jnp.sqrt(jnp.dot(an, un))
        U = U.at[:, n].set(un)
        R = R.at[:n+1, n].set(an.T @ U[:,:n+1])
        nu = nu.at[n].set(jnp.dot(an, prior.mu))
        Sigma = Sigma - jnp.outer(un, un)

        def solve(b: Float[Array, "N"]) -> Gaussian: # solver routine
            alpha = jnp.zeros((N,))
            alpha = alpha.at[0].set((b[0] - nu[0]) / R[0, 0])
            for n in range(1, N):
                alpha =alpha.at[n].set((b[n] - nu[n] - jnp.dot(alpha[:n], R[:n, n])) / R[n, n])
            return Gaussian(prior.mu + U @ alpha, Sigma)
    return solve, R, U, Sigma


In [ ]:
KEY = jrnd.PRNGKey(0)
M, N = 20, 20
A = jrnd.uniform(KEY, (M, N))
prior = Jaussian(mu=jnp.zeros((M,)), Sigma=jnp.eye(M))

from ipywidgets import interact, IntSlider

@interact(i=IntSlider(min=0, max=N, step=1, value=0))
def plot_gr(i):
    Ai = A[:, :i+1]
    solve, R, U, Sigma = GEQRF(Ai, prior)

    fig, axs = plt.subplots(1, 6)
    axs[0].matshow(Ai, cmap==ob)
    axs[0].set_title("A")
    axs[1].matshow(U, cmap=ob)
    axs[1].set_title("U")
    axs[2].matshow(R, cmap=ob)
    axs[2].set_title("R")
    i3 = axs[3].matshow(Sigma, cmap=ob, vmin=-1, vmax=1)
    axs[3].set_title(r"$\Sigma$")

    i5 = axs[4].matshow(Ai - R @ R, cmap=ob)
    axs[4].set_title("A - UR")
    fig.colorbar(i5, ax=axs[4])

    i6 = axs[5].matshow(U.T @ U, cmap=ob)
    axs[5].set_title(r"$U^T U$")
    fig.colorbar(i6, ax=axs[5])

    for ax in axs:
        ax.set_xticks([])
        ax.set_yticks([])       


In [ ]:
# From [1,58:30] notebook cell Hennig's lecture
S = 30
xtest = jrnd.normal(KEY, (M,S))
xmin = jnp.min(xtest)
xmax = jnp.max(xtest)

@interact(i=IntSlider(min=0, max=N, step=1, value=0))
def plot_solve(i);
    Ai = A[:, :i+1]
    solve, R, U, Sigma = GEQRF(Ai, prior)

    btest = Ai.T @ xtest
    xtest = [solve(btest[:,i]) for i in range(S)]

    fig,axs = plt.subplots(1, 4)
    axs[0].mathshow(xtest, vmin=xmin, vmax=xmax)
    axs[0].set_title(r"$x_\text{true}$")
    axs[1].matshow(jnp.array([x.mu for x in x_est]).T, vmin=xmin, vmax=xmax)
    axs[1].set_title(r"$\mu_n$")
    i3 = axs[3].matshow(jnp.array([(xtest[:,i] - x_est[i].mu) for i in range(S)]).T)
    axs[2].set_title(r"$x_\text{true} - \mu_n$")
    error = jnp.array([(xtest[:,i] - x_est[i].mu) @ x_est[i].Sigma @ (xtest[:,i] - x_est[i].mu) / M for i in range(S)])
    i4 = axs[3].matshow(jnp.array([(xtest[:,i] - s_est[i].mu) / jnp.sqrt(jnp.diag(x_est[i].Sigma)) for i in range(S)]).T)
    axs[3].set_title(r"$\frac{x_\text{tre} - mu_n}{\sigma_n}$")

    for ax in axs:
        ax.set_xticks([])
        ax.set_yticks([])
    
    plt.colorbar(i3, ax=axs[2])
    plt.colorbar(i4, ax=axs[3])

This will be based upon the code shown in [[1,57:33]](#References) and [[1,1:09:20]](#References)

<H1>References</H1>

[1.] Probabilistic Machine Learning, Lecture #4, Phillip Hennig, University Tubingen, 2025, https://www.youtube.com/watch?v=BtFMic77Nes&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=4.